# ANN Model Training

This notebook demonstrates the training of the Multilayer Perceptron (MLP) for document classification.

In [ ]:
# Setup
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
import json
from pathlib import Path
from scipy.sparse import load_npz

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks

from src.data_loader import DataLoader
from src.preprocessing import TextPreprocessor
from src.ann_model import DocumentClassifierANN

# Plot styling
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('viridis')
plt.rcParams['figure.figsize'] = (12, 6)

Path('../results/figures').mkdir(parents=True, exist_ok=True)
Path('../results/metrics').mkdir(parents=True, exist_ok=True)
Path('../models').mkdir(parents=True, exist_ok=True)

# Set random seeds for reproducibility
tf.random.set_seed(42)
np.random.seed(42)

## 1. Load Preprocessed Data

In [ ]:
# Load sparse matrices
X_train = load_npz('../data/processed/X_train.npz')
X_val = load_npz('../data/processed/X_val.npz')
X_test = load_npz('../data/processed/X_test.npz')

y_train = np.load('../data/processed/y_train.npy')
y_val = np.load('../data/processed/y_val.npy')
y_test = np.load('../data/processed/y_test.npy')

print(f"Training data: {X_train.shape}")
print(f"Validation data: {X_val.shape}")
print(f"Test data: {X_test.shape}")
print(f"Training labels: {y_train.shape}")
print(f"Number of classes: {len(np.unique(y_train))}")

## 2. Build MLP Model

In [ ]:
# Initialize and build model
ann = DocumentClassifierANN('../config/config.yaml')
input_dim = X_train.shape[1]
model = ann.build_model(input_dim=input_dim)

# Show model summary
model.summary()

## 3. Model Architecture Details

### Architecture:

- **Input Layer**: 15,000 features (TF-IDF)
- **Hidden Layer 1**: 256 neurons, ReLU activation, Dropout 0.40
- **Hidden Layer 2**: 128 neurons, ReLU activation, Dropout 0.30
- **Output Layer**: 20 neurons, Softmax activation

### Training Configuration:

- **Optimizer**: Adam (lr=0.001)
- **Loss**: Sparse Categorical Crossentropy
- **Metrics**: Accuracy
- **Batch Size**: 32
- **Max Epochs**: 40
- **Early Stopping**: Patience 5 (monitor val_loss, restore best weights)
- **LR Scheduler**: ReduceLROnPlateau (factor 0.5, patience 3, min_lr 1e-6)
- **Model Checkpoint**: Save best model by val_accuracy

## 4. Train Model

In [ ]:
# Convert sparse to dense for training (required for Dense layers)
# Note: This requires significant memory (~2-3 GB)
print("Converting sparse matrices to dense...")
X_train_dense = X_train.toarray()
X_val_dense = X_val.toarray()

print(f"Dense training matrix: {X_train_dense.shape} ({X_train_dense.nbytes / 1024**2:.0f} MB)")
print(f"Dense validation matrix: {X_val_dense.shape} ({X_val_dense.nbytes / 1024**2:.0f} MB)")

# Train
import time
start_time = time.time()

history = ann.train(X_train_dense, y_train, X_val_dense, y_val)

training_time = time.time() - start_time
print(f"\nTraining completed in {training_time:.1f} seconds ({training_time/60:.1f} minutes)")

## 5. Training History Visualization

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(1, len(history['accuracy']) + 1)

# Accuracy
axes[0].plot(epochs, history['accuracy'], 'b-', label='Training Accuracy', linewidth=2, marker='o', markersize=4)
axes[0].plot(epochs, history['val_accuracy'], 'r-', label='Validation Accuracy', linewidth=2, marker='s', markersize=4)
axes[0].set_title('Model Accuracy', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim([0, 1.05])

# Loss
axes[1].plot(epochs, history['loss'], 'b-', label='Training Loss', linewidth=2, marker='o', markersize=4)
axes[1].plot(epochs, history['val_loss'], 'r-', label='Validation Loss', linewidth=2, marker='s', markersize=4)
axes[1].set_title('Model Loss', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Loss', fontsize=12)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/figures/training_curves.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# Print training summary
print("=== TRAINING SUMMARY ===")
print(f"Total epochs: {len(history['accuracy'])}")
print(f"Best validation accuracy: {max(history['val_accuracy']):.4f} (epoch {np.argmax(history['val_accuracy']) + 1})")
print(f"Best validation loss: {min(history['val_loss']):.4f} (epoch {np.argmin(history['val_loss']) + 1})")
print(f"Final training accuracy: {history['accuracy'][-1]:.4f}")
print(f"Final validation accuracy: {history['val_accuracy'][-1]:.4f}")
print(f"Final training loss: {history['loss'][-1]:.4f}")
print(f"Final validation loss: {history['val_loss'][-1]:.4f}")
print(f"Training time: {training_time:.1f}s")

## 6. Save Model and Training History

In [ ]:
# Save model
ann.save_model('../models/document_classifier.keras')

# Save training history
history_serializable = {k: [float(v) for v in vals] for k, vals in history.items()}
with open('../results/metrics/training_history.json', 'w') as f:
    json.dump(history_serializable, f, indent=2)

# Save model config
with open('../config/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

with open('../models/model_config.yaml', 'w') as f:
    yaml.dump(config['model'], f, default_flow_style=False)

print("Model and training artifacts saved successfully!")

## 7. Quick Validation Prediction

In [ ]:
# Quick validation check
X_val_dense = X_val.toarray()
val_probs = ann.predict(X_val_dense)
val_preds = np.argmax(val_probs, axis=1)

from sklearn.metrics import accuracy_score, classification_report

val_acc = accuracy_score(y_val, val_preds)
print(f"Validation Accuracy: {val_acc:.4f}")

# Per-class validation accuracy
target_names = [
    'alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware',
    'comp.sys.mac.hardware', 'comp.windows.x', 'misc.forsale', 'rec.autos',
    'rec.motorcycles', 'rec.sport.baseball', 'rec.sport.hockey', 'sci.crypt',
    'sci.electronics', 'sci.med', 'sci.space', 'soc.religion.christian',
    'talk.politics.guns', 'talk.politics.mideast', 'talk.politics.misc', 'talk.religion.misc'
]

report = classification_report(y_val, val_preds, target_names=target_names, output_dict=True)

# Show top/bottom performing categories
cat_f1 = {cat: report[cat]['f1-score'] for cat in target_names if cat in report}
sorted_cats = sorted(cat_f1.items(), key=lambda x: x[1], reverse=True)

print("\nTop 5 categories by F1-score:")
for cat, f1 in sorted_cats[:5]:
    print(f"  {cat}: {f1:.4f}")

print("\nBottom 5 categories by F1-score:")
for cat, f1 in sorted_cats[-5:]:
    print(f"  {cat}: {f1:.4f}")

## 8. Training Complete

### Training Completed Successfully!

**Artifacts Generated:**
- `models/document_classifier.keras` - Trained Keras model
- `models/tfidf_vectorizer.joblib` - Fitted TF-IDF vectorizer (from preprocessing)
- `models/label_encoder.joblib` - Fitted label encoder (from preprocessing)
- `results/metrics/training_history.json` - Training metrics per epoch
- `results/figures/training_curves.png` - Training visualization
- `models/model_config.yaml` - Model configuration

**Next Steps:**
1. Run `src/evaluate.py` for comprehensive test set evaluation
2. Open `notebooks/04_model_evaluation.ipynb` for detailed analysis
3. Launch Streamlit app: `streamlit run app/Home.py`